In [72]:
# !pip install pandas
import pandas as pd

In [73]:
import pandas as pd
import glob

# ── Step 1: Load all 5 years ──────────────────────────────────────────────
files = sorted(glob.glob('./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_20*.csv'))

print(f"Files found: {len(files)}")
for f in files:
    print(f)

df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in files],
    ignore_index=True
)
print(f"\nCombined shape: {df.shape}")

# ── Step 2: Drop null columns only if they exist ──────────────────────────
cols_to_drop = [c for c in ['voltage', 'equipment'] if c in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped: {cols_to_drop}")
else:
    print("voltage/equipment not present — skipped")
print(f"Shape: {df.shape}")

# ── Step 3: Parse datetime (pandas 2.0+ compatible) ──────────────────────
df['datetime_beginning_ept'] = pd.to_datetime(
    df['datetime_beginning_ept']
)
print(f"\nDatetime dtype: {df['datetime_beginning_ept'].dtype}")
print(f"Date range:     {df['datetime_beginning_ept'].min()} "
      f"to {df['datetime_beginning_ept'].max()}")

# ── Step 4: Filter 2019–2023 ─────────────────────────────────────────────
df = df[
    (df['datetime_beginning_ept'] >= '2019-01-01') &
    (df['datetime_beginning_ept'] <  '2024-01-01')
].copy()
print(f"Shape after date filter: {df.shape}")

# ── Step 5: Node types ────────────────────────────────────────────────────
print(f"\nNode types:\n{df['type'].value_counts()}")
print(f"Unique nodes:   {df['pnode_name'].nunique()}")

# ── Step 6: Filter to ZONE + HUB only ────────────────────────────────────
df = df[df['type'].isin(['ZONE', 'HUB'])].copy()
print(f"\nShape after ZONE+HUB filter: {df.shape}")
print(f"Nodes after filter: {df['pnode_name'].nunique()}")
print(f"\nNode names:\n{sorted(df['pnode_name'].unique())}")

# ── Step 7: Select top 50 high-congestion nodes ───────────────────────────
node_congestion = (
    df.groupby('pnode_name')['congestion_price_rt']
      .apply(lambda x: x.abs().mean())
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={'congestion_price_rt': 'mean_abs_congestion'})
)
print(f"\nNodes ranked by mean absolute congestion price:")
print(node_congestion.to_string())

# Take top 50 or all available if fewer than 50
top_n = min(50, len(node_congestion))
top_50_nodes = node_congestion.head(top_n)['pnode_name'].tolist()
print(f"\nSelected {top_n} nodes")

# ── Step 8: Final filtered dataset ───────────────────────────────────────
df_50 = df[df['pnode_name'].isin(top_50_nodes)].copy()
print(f"\nFinal shape: {df_50.shape}")
print(f"Nodes:       {df_50['pnode_name'].nunique()}")
print(f"Date range:  {df_50['datetime_beginning_ept'].min()} "
      f"to {df_50['datetime_beginning_ept'].max()}")

# ── Step 9: Save ──────────────────────────────────────────────────────────
df_50.to_csv('pjm_lmp_top50_2019_2023.csv', index=False)
print("Saved: pjm_lmp_top50_2019_2023.csv")
print(f"File shape: {df_50.shape}")

Files found: 5
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2019.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2020.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2021.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2022.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2023.csv

Combined shape: (516960, 16)
Dropped: ['voltage', 'equipment']
Shape: (516960, 14)


/var/folders/4k/y1z0250n6kx6f5pt734bz7w40000gp/T/ipykernel_16993/3218638638.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime_beginning_ept'] = pd.to_datetime(



Datetime dtype: datetime64[us]
Date range:     2019-02-01 00:00:00 to 2023-12-31 23:00:00
Shape after date filter: (516960, 14)

Node types:
type
HUB    516960
Name: count, dtype: int64
Unique nodes:   12

Shape after ZONE+HUB filter: (516960, 14)
Nodes after filter: 12

Node names:
['AEP GEN HUB', 'AEP-DAYTON HUB', 'ATSI GEN HUB', 'CHICAGO GEN HUB', 'CHICAGO HUB', 'DOMINION HUB', 'EASTERN HUB', 'N ILLINOIS HUB', 'NEW JERSEY HUB', 'OHIO HUB', 'WEST INT HUB', 'WESTERN HUB']

Nodes ranked by mean absolute congestion price:
         pnode_name  mean_abs_congestion
0       EASTERN HUB             9.526464
1    NEW JERSEY HUB             6.103822
2   CHICAGO GEN HUB             4.964611
3       CHICAGO HUB             4.935299
4    N ILLINOIS HUB             4.923383
5      DOMINION HUB             3.924408
6          OHIO HUB             2.581746
7       WESTERN HUB             2.551969
8    AEP-DAYTON HUB             2.458981
9       AEP GEN HUB             2.347192
10     ATSI GEN HUB  

In [74]:
import pandas as pd
import numpy as np

df = pd.read_csv('pjm_lmp_top50_2019_2023.csv')
df['datetime_beginning_ept'] = pd.to_datetime(df['datetime_beginning_ept'])
df = df.sort_values(['pnode_name', 'datetime_beginning_ept']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Nodes: {df['pnode_name'].nunique()}")

# Basic calendar features
df['hour_of_day']    = df['datetime_beginning_ept'].dt.hour
df['day_of_week']    = df['datetime_beginning_ept'].dt.dayofweek   # 0=Monday, 6=Sunday
df['month']          = df['datetime_beginning_ept'].dt.month
df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
df['is_peak_hour']   = df['hour_of_day'].isin(range(7, 23)).astype(int)

# Cyclical encoding — prevents model treating hour 23 and hour 0 as far apart
df['hour_sin']       = np.sin(2 * np.pi * df['hour_of_day'] / 24)
df['hour_cos']       = np.cos(2 * np.pi * df['hour_of_day'] / 24)
df['dow_sin']        = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']        = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin']      = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']      = np.cos(2 * np.pi * df['month'] / 12)

print("Time features added:")
print(df[['datetime_beginning_ept','hour_of_day','day_of_week',
          'hour_sin','hour_cos','dow_sin','dow_cos']].head(5))

# Target column to lag
target = 'total_lmp_rt'

# 24 hourly lags (t-1 to t-24)
lag_hours = list(range(1, 25))

# Also add key seasonal lags
seasonal_lags = [48, 72, 168]  # 2-day, 3-day, 1-week

all_lags = lag_hours + seasonal_lags

for lag in all_lags:
    col_name = f'lmp_lag_{lag}h'
    df[col_name] = (
        df.groupby('pnode_name')[target]
          .shift(lag)
    )

# Verify lag columns created
lag_cols = [c for c in df.columns if 'lmp_lag' in c]
print(f"\nLag features created: {len(lag_cols)}")
print(lag_cols)

# Check NaN count — first 24 rows per node will be NaN (expected)
print(f"\nNaN in lmp_lag_1h:   {df['lmp_lag_1h'].isna().sum()}")
print(f"NaN in lmp_lag_24h:  {df['lmp_lag_24h'].isna().sum()}")
print(f"NaN in lmp_lag_168h: {df['lmp_lag_168h'].isna().sum()}")

# Rolling mean and std per node (captures recent price regime)
for window in [6, 24, 168]:
    df[f'lmp_rollmean_{window}h'] = (
        df.groupby('pnode_name')[target]
          .transform(lambda x: x.shift(1).rolling(window).mean())
    )
    df[f'lmp_rollstd_{window}h'] = (
        df.groupby('pnode_name')[target]
          .transform(lambda x: x.shift(1).rolling(window).std())
    )

# DA-RT spread (very important feature for adversarial manipulation detection)
df['da_rt_spread']         = df['total_lmp_da']        - df['total_lmp_rt']
df['congestion_spread']    = df['congestion_price_da']  - df['congestion_price_rt']
df['energy_spread']        = df['system_energy_price_da'] - df['system_energy_price_rt']

print("Rolling + spread features added")
print(df[['da_rt_spread','congestion_spread','energy_spread']].describe())

# ── Option A: Download EIA natural gas price programmatically ─────────────
df_ng = pd.read_excel('./data/eia_natural_gas_prices.xls',
    sheet_name='Data 1',
    skiprows=2,    # skip first 2 rows
    header=0       # 3rd row becomes header
)



print(df_ng.shape)
print(df_ng.columns.tolist())
print(df_ng.head(5))

# ── Clean and rename columns ──────────────────────────────────────────────
df_ng.columns = ['date', 'ng_price_mmbtu']

# ── Parse date and filter to 2019–2023 ───────────────────────────────────
df_ng['date'] = pd.to_datetime(df_ng['date'])
df_ng = df_ng[
    (df_ng['date'] >= '2019-01-01') &
    (df_ng['date'] <  '2024-01-01')
].copy()

# ── Resample daily → monthly average ─────────────────────────────────────
df_ng = (
    df_ng.set_index('date')
         .resample('MS')          # MS = month start frequency
         ['ng_price_mmbtu']
         .mean()
         .reset_index()
         .rename(columns={'date': 'period'})
)

print(f"\nMonthly averaged shape: {df_ng.shape}")
print(df_ng.head(10))

# ── Add year_month key for merging ────────────────────────────────────────
df_ng['year_month'] = df_ng['period'].dt.to_period('M')

# ── Merge into main dataframe ─────────────────────────────────────────────
df['year_month'] = df['datetime_beginning_ept'].dt.to_period('M')

df = df.merge(
    df_ng[['year_month', 'ng_price_mmbtu']],
    on='year_month',
    how='left'
)

# ── Compute fuel price spread ─────────────────────────────────────────────
heat_rate = 7.5   # MMBtu/MWh — standard CCGT assumption
df['implied_gas_lmp']   = df['ng_price_mmbtu'] * heat_rate
df['fuel_price_spread'] = df['system_energy_price_rt'] - df['implied_gas_lmp']

# ── Verify ────────────────────────────────────────────────────────────────
print(f"\nNulls in ng_price_mmbtu:  {df['ng_price_mmbtu'].isna().sum()}")
print(f"Nulls in fuel_price_spread: {df['fuel_price_spread'].isna().sum()}")
print(f"\nFuel price spread stats:")
print(df['fuel_price_spread'].describe())
print(f"\nSample merged rows:")
print(df[['datetime_beginning_ept','ng_price_mmbtu',
          'implied_gas_lmp','fuel_price_spread']].head(10))

Shape: (516960, 14)
Nodes: 12
Time features added:
  datetime_beginning_ept  hour_of_day  day_of_week  hour_sin  hour_cos  \
0    2019-02-01 00:00:00            0            4  0.000000  1.000000   
1    2019-02-01 01:00:00            1            4  0.258819  0.965926   
2    2019-02-01 02:00:00            2            4  0.500000  0.866025   
3    2019-02-01 03:00:00            3            4  0.707107  0.707107   
4    2019-02-01 04:00:00            4            4  0.866025  0.500000   

    dow_sin   dow_cos  
0 -0.433884 -0.900969  
1 -0.433884 -0.900969  
2 -0.433884 -0.900969  
3 -0.433884 -0.900969  
4 -0.433884 -0.900969  

Lag features created: 27
['lmp_lag_1h', 'lmp_lag_2h', 'lmp_lag_3h', 'lmp_lag_4h', 'lmp_lag_5h', 'lmp_lag_6h', 'lmp_lag_7h', 'lmp_lag_8h', 'lmp_lag_9h', 'lmp_lag_10h', 'lmp_lag_11h', 'lmp_lag_12h', 'lmp_lag_13h', 'lmp_lag_14h', 'lmp_lag_15h', 'lmp_lag_16h', 'lmp_lag_17h', 'lmp_lag_18h', 'lmp_lag_19h', 'lmp_lag_20h', 'lmp_lag_21h', 'lmp_lag_22h', 'lmp_lag_23h

In [80]:
import requests
import pandas as pd
import time
import os

API_KEY = "QUOBHzXLZBJkHBG7Vmldb3gGZvywdrYlghk6Jed3"  # paste your key here

os.makedirs('eia930_interchange', exist_ok=True)

date_chunks = [
    ("2019-01-01T00", "2019-03-31T23"),
    ("2019-04-01T00", "2019-06-30T23"),
    ("2019-07-01T00", "2019-09-30T23"),
    ("2019-10-01T00", "2019-12-31T23"),
    ("2020-01-01T00", "2020-03-31T23"),
    ("2020-04-01T00", "2020-06-30T23"),
    ("2020-07-01T00", "2020-09-30T23"),
    ("2020-10-01T00", "2020-12-31T23"),
    ("2021-01-01T00", "2021-03-31T23"),
    ("2021-04-01T00", "2021-06-30T23"),
    ("2021-07-01T00", "2021-09-30T23"),
    ("2021-10-01T00", "2021-12-31T23"),
    ("2022-01-01T00", "2022-03-31T23"),
    ("2022-04-01T00", "2022-06-30T23"),
    ("2022-07-01T00", "2022-09-30T23"),
    ("2022-10-01T00", "2022-12-31T23"),
    ("2023-01-01T00", "2023-03-31T23"),
    ("2023-04-01T00", "2023-06-30T23"),
    ("2023-07-01T00", "2023-09-30T23"),
    ("2023-10-01T00", "2023-12-31T23"),
]

base_url = "https://api.eia.gov/v2/electricity/rto/interchange-data/data/"
all_chunks = []

for start, end in date_chunks:
    print(f"\nFetching {start} to {end}...")
    offset = 0
    
    while True:
        params = {
            'api_key':              API_KEY,
            'frequency':            'hourly',
            'data[0]':              'value',
            'facets[fromba][]':     'PJM',
            'start':                start,
            'end':                  end,
            'sort[0][column]':      'period',
            'sort[0][direction]':   'asc',
            'offset':               offset,
            'length':               5000
        }
        
        try:
            resp = requests.get(base_url, params=params, timeout=60)
            data = resp.json()
            
            if 'response' not in data:
                print(f"  ERROR — unexpected response: {data}")
                break
            
            rows = data['response']['data']
            
            if not rows:
                print(f"  No more rows at offset {offset}")
                break
            
            chunk = pd.DataFrame(rows)
            all_chunks.append(chunk)
            print(f"  offset {offset}: fetched {len(chunk)} rows "
                  f"— total so far: {sum(len(c) for c in all_chunks)}")
            
            if len(rows) < 5000:
                break  # last page
            
            offset += 5000
            time.sleep(0.5)
            
        except Exception as e:
            print(f"  EXCEPTION at offset {offset}: {e}")
            break

# ── Combine all chunks ────────────────────────────────────────────────────
if all_chunks:
    df_interchange = pd.concat(all_chunks, ignore_index=True)
    
    print(f"\n{'='*50}")
    print(f"DOWNLOAD COMPLETE")
    print(f"{'='*50}")
    print(f"Total rows:    {df_interchange.shape[0]:,}")
    print(f"Total columns: {df_interchange.shape[1]}")
    print(f"Columns:       {df_interchange.columns.tolist()}")
    print(f"\nDate range:    {df_interchange['period'].min()} "
          f"to {df_interchange['period'].max()}")
    print(f"\nSample data:")
    print(df_interchange.head(10))
    
    # ── Save ──────────────────────────────────────────────────────────────
    output_path = 'eia930_interchange/pjm_interchange_2019_2023.csv'
    df_interchange.to_csv(output_path, index=False)
    print(f"\nSaved: {output_path}")
    print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")

else:
    print("No data downloaded — check API key and connection")


Fetching 2019-01-01T00 to 2019-03-31T23...
  offset 0: fetched 5000 rows — total so far: 5000
  offset 5000: fetched 5000 rows — total so far: 10000
  offset 10000: fetched 5000 rows — total so far: 15000
  offset 15000: fetched 120 rows — total so far: 15120

Fetching 2019-04-01T00 to 2019-06-30T23...
  offset 0: fetched 5000 rows — total so far: 20120
  offset 5000: fetched 5000 rows — total so far: 25120
  offset 10000: fetched 5000 rows — total so far: 30120
  offset 15000: fetched 288 rows — total so far: 30408

Fetching 2019-07-01T00 to 2019-09-30T23...
  offset 0: fetched 5000 rows — total so far: 35408
  offset 5000: fetched 5000 rows — total so far: 40408
  offset 10000: fetched 5000 rows — total so far: 45408
  offset 15000: fetched 456 rows — total so far: 45864

Fetching 2019-10-01T00 to 2019-12-31T23...
  offset 0: fetched 5000 rows — total so far: 50864
  offset 5000: fetched 5000 rows — total so far: 55864
  offset 10000: fetched 5000 rows — total so far: 60864
  offset

In [83]:
df_interchange.columns

Index(['period', 'fromba', 'fromba-name', 'toba', 'toba-name', 'value',
       'value-units'],
      dtype='str')

In [84]:
# ── Inspect interchange data ──────────────────────────────────────────────
print(f"Shape: {df_interchange.shape}")
print(f"\nSample rows:")
print(df_interchange.head(5))
print(f"\nUnique fromba: {df_interchange['fromba'].unique()}")
print(f"Unique toba: {df_interchange['toba'].unique()}")
print(f"\nValue stats:")
print(df_interchange['value'].describe())
print(f"\nPeriod sample: {df_interchange['period'].iloc[:5].values}")
print(f"Period dtype: {df_interchange['period'].dtype}")

Shape: (302827, 7)

Sample rows:
          period fromba               fromba-name  toba  \
0  2019-01-01T00    PJM  PJM Interconnection, LLC  CPLE   
1  2019-01-01T00    PJM  PJM Interconnection, LLC  CPLW   
2  2019-01-01T00    PJM  PJM Interconnection, LLC   DUK   
3  2019-01-01T00    PJM  PJM Interconnection, LLC  LGEE   
4  2019-01-01T00    PJM  PJM Interconnection, LLC  MISO   

                                           toba-name  value    value-units  
0                          Duke Energy Progress East    238  megawatthours  
1                          Duke Energy Progress West      0  megawatthours  
2                              Duke Energy Carolinas    167  megawatthours  
3  LG&E and KU Services Company as agent for Loui...    -31  megawatthours  
4     Midcontinent Independent System Operator, Inc.  -2833  megawatthours  

Unique fromba: <StringArray>
['PJM']
Length: 1, dtype: str
Unique toba: <StringArray>
['CPLE', 'CPLW', 'DUK', 'LGEE', 'MISO', 'NYIS', 'TVA']
Length: 

In [85]:
# ── Fix value column — it's string, convert to numeric ───────────────────
df_interchange['value'] = pd.to_numeric(df_interchange['value'], errors='coerce')

# ── Parse period to datetime ──────────────────────────────────────────────
df_interchange['datetime_beginning_ept'] = pd.to_datetime(
    df_interchange['period'].str.replace('T', ' ') + ':00:00'
)

print(f"Datetime sample: {df_interchange['datetime_beginning_ept'].iloc[:3].values}")
print(f"Datetime range:  {df_interchange['datetime_beginning_ept'].min()} "
      f"to {df_interchange['datetime_beginning_ept'].max()}")

# ── Aggregate: sum all BA pairs per hour → single net interchange value ───
hourly_interchange = (
    df_interchange.groupby('datetime_beginning_ept')['value']
    .sum()
    .reset_index()
    .rename(columns={'value': 'net_interchange_mw'})
)

print(f"\nHourly interchange shape: {hourly_interchange.shape}")
print(f"Expected: ~43,824 rows (5 years hourly)")
print(f"\nNet interchange stats:")
print(hourly_interchange['net_interchange_mw'].describe())
print(f"\nSample:")
print(hourly_interchange.head(5))

# ── Add lag features ──────────────────────────────────────────────────────
hourly_interchange = hourly_interchange.sort_values('datetime_beginning_ept')
hourly_interchange['net_interchange_lag1h']  = hourly_interchange['net_interchange_mw'].shift(1)
hourly_interchange['net_interchange_lag24h'] = hourly_interchange['net_interchange_mw'].shift(24)

# ── Merge into main dataframe ─────────────────────────────────────────────
df = df.merge(
    hourly_interchange,
    on='datetime_beginning_ept',
    how='left'
)

print(f"\nAfter merge shape: {df.shape}")
print(f"Nulls in net_interchange_mw: {df['net_interchange_mw'].isna().sum()}")
print(f"\nSample merged:")
print(df[['datetime_beginning_ept', 'pnode_name',
          'total_lmp_rt', 'net_interchange_mw',
          'net_interchange_lag1h', 'net_interchange_lag24h']].head(10))
          

Datetime sample: ['2019-01-01T00:00:00.000000' '2019-01-01T00:00:00.000000'
 '2019-01-01T00:00:00.000000']
Datetime range:  2019-01-01 00:00:00 to 2023-12-31 23:00:00

Hourly interchange shape: (43652, 2)
Expected: ~43,824 rows (5 years hourly)

Net interchange stats:
count    43652.000000
mean      4087.752130
std       5603.818046
min     -15692.000000
25%       1442.500000
50%       4998.500000
75%       7829.250000
max      29063.000000
Name: net_interchange_mw, dtype: float64

Sample:
  datetime_beginning_ept  net_interchange_mw
0    2019-01-01 00:00:00               -3370
1    2019-01-01 01:00:00               -3639
2    2019-01-01 02:00:00               -2718
3    2019-01-01 03:00:00               -2398
4    2019-01-01 04:00:00               -3209

After merge shape: (516960, 68)
Nulls in net_interchange_mw: 2076

Sample merged:
  datetime_beginning_ept   pnode_name  total_lmp_rt  net_interchange_mw  \
0    2019-02-01 00:00:00  AEP GEN HUB     42.438991             -1029.0   
1 

In [87]:
# ── Check total nulls before dropna ──────────────────────────────────────
print(f"Shape before dropna: {df.shape}")
print(f"\nNull counts (only columns with nulls):")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

# ── Drop rows where lag features are NaN ─────────────────────────────────
lag_cols = [c for c in df.columns if any(x in c for x in 
            ['lmp_lag', 'rollmean', 'rollstd'])]

df_clean = df.dropna(subset=lag_cols).reset_index(drop=True)

print(f"\nShape after dropna: {df_clean.shape}")
print(f"Rows dropped:       {df.shape[0] - df_clean.shape[0]}")

# ── Fill remaining NaN in interchange with forward fill ──────────────────
df_clean['net_interchange_mw']     = df_clean['net_interchange_mw'].ffill()
df_clean['net_interchange_lag1h']  = df_clean['net_interchange_lag1h'].ffill()
df_clean['net_interchange_lag24h'] = df_clean['net_interchange_lag24h'].ffill()

print(f"\nNulls after ffill:")
print(df_clean[['net_interchange_mw',
                'net_interchange_lag1h',
                'net_interchange_lag24h']].isnull().sum())

# ── Final feature summary ─────────────────────────────────────────────────
feature_groups = {
    'Autoregressive lags':  [c for c in df_clean.columns if 'lmp_lag' in c],
    'Rolling statistics':   [c for c in df_clean.columns if 'roll' in c],
    'Time features':        [c for c in df_clean.columns if any(x in c for x in
                             ['hour', 'dow', 'month', 'weekend', 'peak'])],
    'Spread features':      ['da_rt_spread', 'congestion_spread',
                             'energy_spread', 'fuel_price_spread'],
    'Gas price':            ['ng_price_mmbtu', 'implied_gas_lmp'],
    'Interchange':          ['net_interchange_mw', 'net_interchange_lag1h',
                             'net_interchange_lag24h'],
    'Raw LMP components':   ['total_lmp_rt', 'system_energy_price_rt',
                             'congestion_price_rt', 'marginal_loss_price_rt',
                             'total_lmp_da', 'system_energy_price_da',
                             'congestion_price_da', 'marginal_loss_price_da'],
}

print("\n" + "="*45)
print("FEATURE ENGINEERING COMPLETE")
print("="*45)
total = 0
for group, cols in feature_groups.items():
    available = [c for c in cols if c in df_clean.columns]
    print(f"  {group:25s}: {len(available)} features")
    total += len(available)
print(f"  {'─'*38}")
print(f"  {'Total':25s}: {total} features")
print(f"  {'Rows':25s}: {df_clean.shape[0]:,}")
print(f"  {'Nodes':25s}: {df_clean['pnode_name'].nunique()}")
print(f"  {'Date range':25s}: {df_clean['datetime_beginning_ept'].min()} "
      f"to {df_clean['datetime_beginning_ept'].max()}")

# ── Save final engineered dataset ─────────────────────────────────────────
df_clean.to_csv('pjm_features_engineered.csv', index=False)
print(f"\nSaved: pjm_features_engineered.csv")
print(f"Final shape: {df_clean.shape}")


Shape before dropna: (516960, 68)

Null counts (only columns with nulls):
zone                      516960
lmp_lag_1h                    12
lmp_lag_2h                    24
lmp_lag_3h                    36
lmp_lag_4h                    48
lmp_lag_5h                    60
lmp_lag_6h                    72
lmp_lag_7h                    84
lmp_lag_8h                    96
lmp_lag_9h                   108
lmp_lag_10h                  120
lmp_lag_11h                  132
lmp_lag_12h                  144
lmp_lag_13h                  156
lmp_lag_14h                  168
lmp_lag_15h                  180
lmp_lag_16h                  192
lmp_lag_17h                  204
lmp_lag_18h                  216
lmp_lag_19h                  228
lmp_lag_20h                  240
lmp_lag_21h                  252
lmp_lag_22h                  264
lmp_lag_23h                  276
lmp_lag_24h                  288
lmp_lag_48h                  576
lmp_lag_72h                  864
lmp_lag_168h                2016
lm

In [88]:
!pip install torch scikit-learn numpy pandas matplotlib seaborn tqdm

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 7.3 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 11.3 MB/

In [89]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import os
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Config ────────────────────────────────────────────────────────────────
CFG = {
    # Data
    'data_path':      'pjm_features_engineered.csv',
    'targets':        ['total_lmp_rt', 'total_lmp_da'],
    'seq_len':        24,        # 24-hour input window
    'pred_len':       1,         # 1-hour ahead forecast

    # Features to use
    'feature_cols':   None,      # set below after loading

    # WGAN-GP
    'noise_dim':      32,        # reduced for CPU
    'gen_hidden':     64,
    'disc_hidden':    64,
    'n_critic':       3,         # discriminator updates per generator update
    'gp_lambda':      10,        # gradient penalty weight
    'gen_lr':         2e-4,
    'disc_lr':        2e-4,
    'gan_epochs':     30,        # reduced for CPU feasibility
    'gan_batch':      256,

    # LSTM forecaster
    'lstm_hidden':    64,
    'lstm_layers':    2,
    'lstm_lr':        1e-3,
    'lstm_epochs':    20,
    'lstm_batch':     256,
    'dropout':        0.2,

    # Randomized smoothing
    'sigma':          0.05,      # noise std for smoothing
    'n_smooth':       64,        # samples for smoothed inference (CPU-friendly)

    # Adversarial attack (PGD)
    'pgd_eps':        [0.01, 0.05, 0.10],
    'pgd_steps':      10,
    'pgd_alpha':      0.005,

    # Output
    'output_dir':     'robustlmp_outputs',
}

os.makedirs(CFG['output_dir'], exist_ok=True)
print("Config loaded")
print(f"Device: CPU")
print(f"Targets: {CFG['targets']}")

Config loaded
Device: CPU
Targets: ['total_lmp_rt', 'total_lmp_da']


In [90]:
# ── Load data ─────────────────────────────────────────────────────────────
df = pd.read_csv(CFG['data_path'])
df['datetime_beginning_ept'] = pd.to_datetime(df['datetime_beginning_ept'])
df = df.sort_values(['pnode_name', 'datetime_beginning_ept']).reset_index(drop=True)

print(f"Loaded: {df.shape}")
print(f"Nodes:  {df['pnode_name'].unique()}")

# ── Define feature columns ────────────────────────────────────────────────
exclude_cols = [
    'datetime_beginning_utc', 'datetime_beginning_ept',
    'pnode_id', 'pnode_name', 'type', 'zone', 'year_month',
    'total_lmp_rt', 'total_lmp_da'   # targets — not input features
]

feature_cols = [c for c in df.columns if c not in exclude_cols]
CFG['feature_cols'] = feature_cols

print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols)

# ── Train/val/test split (temporal — no leakage) ──────────────────────────
# Train: 2019-02 to 2022-12 (~80%)
# Val:   2023-01 to 2023-06 (~10%)
# Test:  2023-07 to 2023-12 (~10%)

train_mask = df['datetime_beginning_ept'] <  '2023-01-01'
val_mask   = (df['datetime_beginning_ept'] >= '2023-01-01') & \
             (df['datetime_beginning_ept'] <  '2023-07-01')
test_mask  = df['datetime_beginning_ept'] >= '2023-07-01'

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print(f"\nTrain: {df_train.shape} | "
      f"Val: {df_val.shape} | "
      f"Test: {df_test.shape}")

# ── Scale features ────────────────────────────────────────────────────────
# Fit scaler on train only — prevent data leakage
scaler_X = MinMaxScaler()
scaler_rt = MinMaxScaler()
scaler_da = MinMaxScaler()

X_train = scaler_X.fit_transform(df_train[feature_cols].fillna(0))
X_val   = scaler_X.transform(df_val[feature_cols].fillna(0))
X_test  = scaler_X.transform(df_test[feature_cols].fillna(0))

y_train_rt = scaler_rt.fit_transform(df_train[['total_lmp_rt']])
y_val_rt   = scaler_rt.transform(df_val[['total_lmp_rt']])
y_test_rt  = scaler_rt.transform(df_test[['total_lmp_rt']])

y_train_da = scaler_da.fit_transform(df_train[['total_lmp_da']])
y_val_da   = scaler_da.transform(df_val[['total_lmp_da']])
y_test_da  = scaler_da.transform(df_test[['total_lmp_da']])

print(f"\nScaling complete")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

Loaded: (514944, 68)
Nodes:  <StringArray>
[    'AEP GEN HUB',  'AEP-DAYTON HUB',    'ATSI GEN HUB', 'CHICAGO GEN HUB',
     'CHICAGO HUB',    'DOMINION HUB',     'EASTERN HUB',  'N ILLINOIS HUB',
  'NEW JERSEY HUB',        'OHIO HUB',    'WEST INT HUB',     'WESTERN HUB']
Length: 12, dtype: str

Feature columns (59):
['system_energy_price_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'system_energy_price_da', 'congestion_price_da', 'marginal_loss_price_da', 'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'lmp_lag_1h', 'lmp_lag_2h', 'lmp_lag_3h', 'lmp_lag_4h', 'lmp_lag_5h', 'lmp_lag_6h', 'lmp_lag_7h', 'lmp_lag_8h', 'lmp_lag_9h', 'lmp_lag_10h', 'lmp_lag_11h', 'lmp_lag_12h', 'lmp_lag_13h', 'lmp_lag_14h', 'lmp_lag_15h', 'lmp_lag_16h', 'lmp_lag_17h', 'lmp_lag_18h', 'lmp_lag_19h', 'lmp_lag_20h', 'lmp_lag_21h', 'lmp_lag_22h', 'lmp_lag_23h', 'lmp_lag_24h', 'lmp_lag_48h', 'lmp_lag_72h', 'lmp_la

In [91]:
class LMPDataset(Dataset):
    def __init__(self, X, y_rt, y_da, seq_len=24):
        self.X      = torch.FloatTensor(X)
        self.y_rt   = torch.FloatTensor(y_rt)
        self.y_da   = torch.FloatTensor(y_da)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        x_seq  = self.X[idx : idx + self.seq_len]           # (seq_len, n_features)
        y_rt   = self.y_rt[idx + self.seq_len]              # (1,)
        y_da   = self.y_da[idx + self.seq_len]              # (1,)
        return x_seq, y_rt, y_da

train_ds = LMPDataset(X_train, y_train_rt, y_train_da, CFG['seq_len'])
val_ds   = LMPDataset(X_val,   y_val_rt,   y_val_da,   CFG['seq_len'])
test_ds  = LMPDataset(X_test,  y_test_rt,  y_test_da,  CFG['seq_len'])

train_loader = DataLoader(train_ds, batch_size=CFG['lstm_batch'],
                          shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['lstm_batch'],
                          shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=CFG['lstm_batch'],
                          shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 1600
Val batches:   204
Test batches:  207


In [ ]:
import pandas as pd
import glob

# ── Step 1: Load all 5 years ──────────────────────────────────────────────
files = sorted(glob.glob('./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_20*.csv'))

print(f"Files found: {len(files)}")
for f in files:
    print(f)

df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in files],
    ignore_index=True
)
print(f"\nCombined shape: {df.shape}")

# ── Step 2: Drop null columns only if they exist ──────────────────────────
cols_to_drop = [c for c in ['voltage', 'equipment'] if c in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped: {cols_to_drop}")
else:
    print("voltage/equipment not present — skipped")
print(f"Shape: {df.shape}")

# ── Step 3: Parse datetime (pandas 2.0+ compatible) ──────────────────────
df['datetime_beginning_ept'] = pd.to_datetime(
    df['datetime_beginning_ept']
)
print(f"\nDatetime dtype: {df['datetime_beginning_ept'].dtype}")
print(f"Date range:     {df['datetime_beginning_ept'].min()} "
      f"to {df['datetime_beginning_ept'].max()}")

# ── Step 4: Filter 2019–2023 ─────────────────────────────────────────────
df = df[
    (df['datetime_beginning_ept'] >= '2019-01-01') &
    (df['datetime_beginning_ept'] <  '2024-01-01')
].copy()
print(f"Shape after date filter: {df.shape}")

# ── Step 5: Node types ────────────────────────────────────────────────────
print(f"\nNode types:\n{df['type'].value_counts()}")
print(f"Unique nodes:   {df['pnode_name'].nunique()}")

# ── Step 6: Filter to ZONE + HUB only ────────────────────────────────────
df = df[df['type'].isin(['ZONE', 'HUB'])].copy()
print(f"\nShape after ZONE+HUB filter: {df.shape}")
print(f"Nodes after filter: {df['pnode_name'].nunique()}")
print(f"\nNode names:\n{sorted(df['pnode_name'].unique())}")

# ── Step 7: Select top 50 high-congestion nodes ───────────────────────────
node_congestion = (
    df.groupby('pnode_name')['congestion_price_rt']
      .apply(lambda x: x.abs().mean())
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={'congestion_price_rt': 'mean_abs_congestion'})
)
print(f"\nNodes ranked by mean absolute congestion price:")
print(node_congestion.to_string())

# Take top 50 or all available if fewer than 50
top_n = min(50, len(node_congestion))
top_50_nodes = node_congestion.head(top_n)['pnode_name'].tolist()
print(f"\nSelected {top_n} nodes")

# ── Step 8: Final filtered dataset ───────────────────────────────────────
df_50 = df[df['pnode_name'].isin(top_50_nodes)].copy()
print(f"\nFinal shape: {df_50.shape}")
print(f"Nodes:       {df_50['pnode_name'].nunique()}")
print(f"Date range:  {df_50['datetime_beginning_ept'].min()} "
      f"to {df_50['datetime_beginning_ept'].max()}")

# ── Step 9: Save ──────────────────────────────────────────────────────────
df_50.to_csv('pjm_lmp_top50_2019_2023.csv', index=False)
print("Saved: pjm_lmp_top50_2019_2023.csv")
print(f"File shape: {df_50.shape}")

Files found: 5
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2019.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2020.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2021.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2022.csv
./data/PJM hourly LMP and load data 2019–2023/rt_da_monthly_lmps_2023.csv

Combined shape: (516960, 16)
Dropped: ['voltage', 'equipment']
Shape: (516960, 14)


/var/folders/4k/y1z0250n6kx6f5pt734bz7w40000gp/T/ipykernel_16993/3218638638.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime_beginning_ept'] = pd.to_datetime(



Datetime dtype: datetime64[us]
Date range:     2019-02-01 00:00:00 to 2023-12-31 23:00:00
Shape after date filter: (516960, 14)

Node types:
type
HUB    516960
Name: count, dtype: int64
Unique nodes:   12

Shape after ZONE+HUB filter: (516960, 14)
Nodes after filter: 12

Node names:
['AEP GEN HUB', 'AEP-DAYTON HUB', 'ATSI GEN HUB', 'CHICAGO GEN HUB', 'CHICAGO HUB', 'DOMINION HUB', 'EASTERN HUB', 'N ILLINOIS HUB', 'NEW JERSEY HUB', 'OHIO HUB', 'WEST INT HUB', 'WESTERN HUB']

Nodes ranked by mean absolute congestion price:
         pnode_name  mean_abs_congestion
0       EASTERN HUB             9.526464
1    NEW JERSEY HUB             6.103822
2   CHICAGO GEN HUB             4.964611
3       CHICAGO HUB             4.935299
4    N ILLINOIS HUB             4.923383
5      DOMINION HUB             3.924408
6          OHIO HUB             2.581746
7       WESTERN HUB             2.551969
8    AEP-DAYTON HUB             2.458981
9       AEP GEN HUB             2.347192
10     ATSI GEN HUB  

In [105]:
import requests
import pandas as pd
import time
import os

NOAA_TOKEN = "OoCkGhGuTEWvluWNFlhnSpqtnOYvQcvP"  # paste here

stations = {
    'Philadelphia':  'GHCND:USW00013739',
    'Pittsburgh':    'GHCND:USW00094823',
    'Chicago':       'GHCND:USW00094846',
    'Columbus_OH':   'GHCND:USW00014821',
    'Washington_DC': 'GHCND:USW00013743',
}

os.makedirs('noaa_weather', exist_ok=True)
headers  = {'token': NOAA_TOKEN}
base_url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"

all_weather = []

for city, station_id in stations.items():
    print(f"\nDownloading {city}...")

    for year in range(2019, 2024):
        offset = 0

        while True:
            params = {
                'datasetid':  'GHCND',
                'stationid':  station_id,
                'datatypeid': ['TMAX', 'TMIN', 'AWND', 'PRCP'],
                'startdate':  f'{year}-01-01',
                'enddate':    f'{year}-12-31',
                'limit':      1000,
                'offset':     offset,
                'units':      'metric',
            }

            try:
                resp = requests.get(base_url, headers=headers,
                                    params=params, timeout=30)
                data = resp.json()

                if 'results' not in data:
                    print(f"  {year} offset {offset}: no results")
                    break

                rows = data['results']
                df_chunk = pd.DataFrame(rows)
                df_chunk['city'] = city
                all_weather.append(df_chunk)

                total_available = data['metadata']['resultset']['count']
                fetched_so_far  = offset + len(rows)

                print(f"  {year} offset {offset:4d}: "
                      f"{len(rows)} rows | "
                      f"{fetched_so_far}/{total_available} total")

                if fetched_so_far >= total_available:
                    break

                offset += 1000
                time.sleep(0.3)

            except Exception as e:
                print(f"  {year} offset {offset}: error — {e}")
                break

# ── Combine and save ──────────────────────────────────────────────────────
df_weather_raw = pd.concat(all_weather, ignore_index=True)
print(f"\nRaw weather shape: {df_weather_raw.shape}")
print(f"Expected ~:        {5 * 5 * 365 * 4} records "
      f"(5 cities × 5 years × 365 days × 4 variables)")

df_weather_raw.to_csv('noaa_weather/pjm_weather_raw_2019_2023.csv', index=False)
print("Saved: pjm_weather_raw_2019_2023.csv")



  2019 offset    0: 1000 rows | 1000/1460 total
  2019 offset 1000: 461 rows | 1461/1460 total
  2020 offset    0: 1000 rows | 1000/1464 total
  2020 offset 1000: 465 rows | 1465/1464 total
  2021 offset    0: 1000 rows | 1000/1460 total
  2021 offset 1000: 461 rows | 1461/1460 total
  2022 offset    0: 1000 rows | 1000/1460 total
  2022 offset 1000: 461 rows | 1461/1460 total
  2023 offset    0: 1000 rows | 1000/1460 total
  2023 offset 1000: 461 rows | 1461/1460 total

  2019 offset    0: 1000 rows | 1000/1460 total
  2019 offset 1000: 461 rows | 1461/1460 total
  2020 offset    0: 1000 rows | 1000/1464 total
  2020 offset 1000: 465 rows | 1465/1464 total
  2021 offset    0: 1000 rows | 1000/1460 total
  2021 offset 1000: 461 rows | 1461/1460 total
  2022 offset    0: 1000 rows | 1000/1460 total
  2022 offset 1000: 461 rows | 1461/1460 total
  2023 offset    0: 1000 rows | 1000/1460 total
  2023 offset 1000: 461 rows | 1461/1460 total

  2019 offset    0: 1000 rows | 1000/1460 total

In [106]:
import pandas as pd
import numpy as np

# ── Load raw weather ──────────────────────────────────────────────────────
df_w = pd.read_csv('noaa_weather/pjm_weather_raw_2019_2023.csv')

print(f"Shape: {df_w.shape}")
print(f"Datatypes: {df_w['datatype'].unique()}")

# ── Pivot: one row per date per city ──────────────────────────────────────
df_w['date'] = pd.to_datetime(df_w['date']).dt.normalize()

df_pivot = df_w.pivot_table(
    index   = ['date', 'city'],
    columns = 'datatype',
    values  = 'value',
    aggfunc = 'mean'
).reset_index()

df_pivot.columns.name = None
print(f"\nPivoted shape: {df_pivot.shape}")
print(df_pivot.head(3))

# ── Average across all 5 cities → single daily PJM weather ───────────────
df_daily = (
    df_pivot.groupby('date')
    .agg(
        temp_max   = ('TMAX', 'mean'),
        temp_min   = ('TMIN', 'mean'),
        wind_speed = ('AWND', 'mean'),
        precip     = ('PRCP', 'mean')
    )
    .reset_index()
)

df_daily['temp_avg'] = (df_daily['temp_max'] + df_daily['temp_min']) / 2

# Heating/Cooling Degree Days
base_temp = 18.3
df_daily['HDD'] = (base_temp - df_daily['temp_avg']).clip(lower=0)
df_daily['CDD'] = (df_daily['temp_avg'] - base_temp).clip(lower=0)

print(f"\nDaily weather shape: {df_daily.shape}")
print(f"Date range: {df_daily['date'].min()} to {df_daily['date'].max()}")
print(f"Temp range: {df_daily['temp_avg'].min():.1f}°C "
      f"to {df_daily['temp_avg'].max():.1f}°C")
print(f"Missing days: {df_daily.isnull().sum().sum()}")

# ── Expand daily → hourly ─────────────────────────────────────────────────
# Vectorised — much faster than row-by-row loop
df_daily_rep = df_daily.loc[
    df_daily.index.repeat(24)
].reset_index(drop=True)

hours = list(range(24)) * len(df_daily)
df_daily_rep['hour'] = hours

df_daily_rep['datetime_beginning_ept'] = (
    df_daily_rep['date'] +
    pd.to_timedelta(df_daily_rep['hour'], unit='h')
)

df_hourly_weather = df_daily_rep.drop(columns=['date', 'hour'])

print(f"\nHourly weather shape: {df_hourly_weather.shape}")
print(f"Expected:             43,848 rows")
print(f"Date range: {df_hourly_weather['datetime_beginning_ept'].min()} "
      f"to {df_hourly_weather['datetime_beginning_ept'].max()}")

df_hourly_weather.to_csv(
    'noaa_weather/pjm_weather_hourly_2019_2023.csv', index=False)
print("Saved: pjm_weather_hourly_2019_2023.csv")

# ── Merge into engineered features ───────────────────────────────────────
df_clean = pd.read_csv('pjm_features_engineered.csv')
df_clean['datetime_beginning_ept'] = pd.to_datetime(
    df_clean['datetime_beginning_ept'])

df_clean = df_clean.merge(
    df_hourly_weather,
    on  = 'datetime_beginning_ept',
    how = 'left'
)

weather_cols = ['temp_avg', 'temp_max', 'temp_min',
                'wind_speed', 'precip', 'HDD', 'CDD']

# Forward fill any remaining gaps
df_clean[weather_cols] = df_clean[weather_cols].ffill()

print(f"\nAfter weather merge: {df_clean.shape}")
print(f"Weather nulls:       {df_clean[weather_cols].isnull().sum().sum()}")
print(f"\nWeather sample:")
print(df_clean[['datetime_beginning_ept'] + weather_cols].head(5))

# ── Save final dataset ────────────────────────────────────────────────────
df_clean.to_csv('pjm_features_engineered_weather.csv', index=False)
print(f"\nSaved: pjm_features_engineered_weather.csv")
print(f"Final shape:    {df_clean.shape}")
print(f"Total features: {df_clean.shape[1]}")

# ── Verify missing hours fixed ────────────────────────────────────────────
full_range = pd.date_range(start='2019-02-08', end='2023-12-31', freq='h')
missing = full_range.difference(
    df_hourly_weather['datetime_beginning_ept'])
print(f"\nMissing hours in weather: {len(missing)}")
print(f"Expected: ~0")

Shape: (36542, 6)
Datatypes: <StringArray>
['AWND', 'PRCP', 'TMAX', 'TMIN']
Length: 4, dtype: str

Pivoted shape: (9130, 6)
        date          city  AWND  PRCP  TMAX  TMIN
0 2019-01-01       Chicago   3.4   0.3   1.1  -3.2
1 2019-01-01   Columbus_OH   4.6   0.0  14.4   3.9
2 2019-01-01  Philadelphia   7.0   2.3  16.1   6.1

Daily weather shape: (1826, 8)
Date range: 2019-01-01 00:00:00 to 2023-12-31 00:00:00
Temp range: -15.0°C to 29.9°C
Missing days: 0

Hourly weather shape: (43824, 8)
Expected:             43,848 rows
Date range: 2019-01-01 00:00:00 to 2023-12-31 23:00:00
Saved: pjm_weather_hourly_2019_2023.csv

After weather merge: (514944, 75)
Weather nulls:       0

Weather sample:
  datetime_beginning_ept  temp_avg  temp_max  temp_min  wind_speed  precip  \
0    2019-02-08 00:00:00       1.2      7.68     -5.28        7.12     0.6   
1    2019-02-08 01:00:00       1.2      7.68     -5.28        7.12     0.6   
2    2019-02-08 02:00:00       1.2      7.68     -5.28        7.12 

In [107]:
# ── Reload with weather features ──────────────────────────────────────────
df_clean = pd.read_csv('pjm_features_engineered_weather.csv')
df_clean['datetime_beginning_ept'] = pd.to_datetime(
    df_clean['datetime_beginning_ept'])
df_clean = df_clean.sort_values(
    ['pnode_name', 'datetime_beginning_ept']).reset_index(drop=True)

print(f"Loaded: {df_clean.shape}")

# ── Feature columns ───────────────────────────────────────────────────────
exclude_cols = [
    'datetime_beginning_utc', 'datetime_beginning_ept',
    'pnode_id', 'pnode_name', 'type', 'zone', 'year_month',
    'total_lmp_rt', 'total_lmp_da'
]
feature_cols = [c for c in df_clean.columns if c not in exclude_cols]
n_features   = len(feature_cols)

print(f"Features: {n_features}")
print(feature_cols)

# ── CPU-optimised config ──────────────────────────────────────────────────
CFG['gan_epochs']  = 15
CFG['lstm_epochs'] = 15
CFG['gan_batch']   = 512
CFG['lstm_batch']  = 512
CFG['n_critic']    = 2
CFG['n_smooth']    = 16

# ── Train/val/test split ──────────────────────────────────────────────────
train_mask = df_clean['datetime_beginning_ept'] <  '2023-01-01'
val_mask   = (df_clean['datetime_beginning_ept'] >= '2023-01-01') & \
             (df_clean['datetime_beginning_ept'] <  '2023-07-01')
test_mask  = df_clean['datetime_beginning_ept'] >= '2023-07-01'

df_train = df_clean[train_mask].copy()
df_val   = df_clean[val_mask].copy()
df_test  = df_clean[test_mask].copy()

print(f"\nTrain: {df_train.shape} | Val: {df_val.shape} | Test: {df_test.shape}")

# ── Scale ─────────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler

scaler_X  = MinMaxScaler()
scaler_rt = MinMaxScaler()
scaler_da = MinMaxScaler()

X_train = scaler_X.fit_transform(df_train[feature_cols].fillna(0))
X_val   = scaler_X.transform(df_val[feature_cols].fillna(0))
X_test  = scaler_X.transform(df_test[feature_cols].fillna(0))

y_train_rt = scaler_rt.fit_transform(df_train[['total_lmp_rt']])
y_val_rt   = scaler_rt.transform(df_val[['total_lmp_rt']])
y_test_rt  = scaler_rt.transform(df_test[['total_lmp_rt']])

y_train_da = scaler_da.fit_transform(df_train[['total_lmp_da']])
y_val_da   = scaler_da.transform(df_val[['total_lmp_da']])
y_test_da  = scaler_da.transform(df_test[['total_lmp_da']])

print("Scaling complete")

# ── Datasets and loaders ──────────────────────────────────────────────────
train_ds = LMPDataset(X_train, y_train_rt, y_train_da, CFG['seq_len'])
val_ds   = LMPDataset(X_val,   y_val_rt,   y_val_da,   CFG['seq_len'])
test_ds  = LMPDataset(X_test,  y_test_rt,  y_test_da,  CFG['seq_len'])

train_loader = DataLoader(train_ds, batch_size=CFG['lstm_batch'],
                          shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['lstm_batch'],
                          shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=CFG['lstm_batch'],
                          shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# ── Reinitialise models ───────────────────────────────────────────────────
G = Generator(CFG['noise_dim'], n_features,
              CFG['seq_len'],   CFG['gen_hidden'])
D = Discriminator(n_features, CFG['seq_len'], CFG['disc_hidden'])

opt_G = torch.optim.Adam(G.parameters(), lr=CFG['gen_lr'],  betas=(0.0, 0.9))
opt_D = torch.optim.Adam(D.parameters(), lr=CFG['disc_lr'], betas=(0.0, 0.9))

print(f"\nn_features:           {n_features}")
print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")
print(f"\nEstimated WGAN time:  ~45–60 mins on CPU")
print(f"Estimated LSTM time:  ~20–30 mins on CPU")
print(f"Total estimated:      ~65–90 mins")

Loaded: (514944, 75)
Features: 66
['system_energy_price_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'system_energy_price_da', 'congestion_price_da', 'marginal_loss_price_da', 'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'lmp_lag_1h', 'lmp_lag_2h', 'lmp_lag_3h', 'lmp_lag_4h', 'lmp_lag_5h', 'lmp_lag_6h', 'lmp_lag_7h', 'lmp_lag_8h', 'lmp_lag_9h', 'lmp_lag_10h', 'lmp_lag_11h', 'lmp_lag_12h', 'lmp_lag_13h', 'lmp_lag_14h', 'lmp_lag_15h', 'lmp_lag_16h', 'lmp_lag_17h', 'lmp_lag_18h', 'lmp_lag_19h', 'lmp_lag_20h', 'lmp_lag_21h', 'lmp_lag_22h', 'lmp_lag_23h', 'lmp_lag_24h', 'lmp_lag_48h', 'lmp_lag_72h', 'lmp_lag_168h', 'lmp_rollmean_6h', 'lmp_rollstd_6h', 'lmp_rollmean_24h', 'lmp_rollstd_24h', 'lmp_rollmean_168h', 'lmp_rollstd_168h', 'da_rt_spread', 'congestion_spread', 'energy_spread', 'ng_price_mmbtu', 'implied_gas_lmp', 'fuel_price_spread', 'net_interchange_mw', 'net_interchange_lag1h', '

In [111]:
from tqdm import tqdm
import time
import os

# ── Redefine gradient penalty ─────────────────────────────────────────────
def gradient_penalty(disc, real, fake):
    batch = real.size(0)
    alpha = torch.rand(batch, 1, 1)
    alpha = alpha.expand_as(real)

    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = disc(interpolated)

    gradients = torch.autograd.grad(
        outputs      = d_interp,
        inputs       = interpolated,
        grad_outputs = torch.ones_like(d_interp),
        create_graph = True,
        retain_graph = True
    )[0]

    gradients = gradients.reshape(batch, -1)
    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gp


# ── Checkpoint save/load helpers ──────────────────────────────────────────
def save_checkpoint(epoch, G, D, opt_G, opt_D,
                    g_losses, d_losses, path):
    torch.save({
        'epoch':    epoch,
        'G_state':  G.state_dict(),
        'D_state':  D.state_dict(),
        'optG':     opt_G.state_dict(),
        'optD':     opt_D.state_dict(),
        'g_losses': g_losses,
        'd_losses': d_losses,
    }, path)
    print(f"  ✅ Checkpoint saved: {path}")


def load_checkpoint(path, G, D, opt_G, opt_D):
    ckpt = torch.load(path)
    G.load_state_dict(ckpt['G_state'])
    D.load_state_dict(ckpt['D_state'])
    opt_G.load_state_dict(ckpt['optG'])
    opt_D.load_state_dict(ckpt['optD'])
    print(f"Resumed from epoch {ckpt['epoch'] + 1}")
    return (ckpt['epoch'] + 1,
            ckpt['g_losses'],
            ckpt['d_losses'])


# ── Check for existing checkpoint to resume from ─────────────────────────
checkpoint_dir  = f"{CFG['output_dir']}/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

latest_ckpt = None
start_epoch = 0
g_losses    = []
d_losses    = []

# Find latest checkpoint
ckpt_files = sorted([
    f for f in os.listdir(checkpoint_dir)
    if f.startswith('wgan_epoch_') and f.endswith('.pt')
])

if ckpt_files:
    latest_ckpt = os.path.join(checkpoint_dir, ckpt_files[-1])
    print(f"Found checkpoint: {latest_ckpt}")
    print("Resume from checkpoint? Set resume=True to continue")
    resume = True  # set to False to start fresh

    if resume:
        start_epoch, g_losses, d_losses = load_checkpoint(
            latest_ckpt, G, D, opt_G, opt_D)
else:
    print("No checkpoint found — starting fresh")

print(f"Starting from epoch: {start_epoch + 1}")
print(f"Remaining epochs:    {CFG['gan_epochs'] - start_epoch}")


# ── Training loop with checkpoints ───────────────────────────────────────
start_time = time.time()

print(f"\nTraining WGAN-GP...")
print(f"Epochs: {CFG['gan_epochs']} | Batch: {CFG['gan_batch']}")
print(f"n_critic: {CFG['n_critic']} | GP lambda: {CFG['gp_lambda']}")
print(f"n_features: {n_features}")
print(f"Checkpoints every 5 epochs → {checkpoint_dir}\n")

epoch_pbar = tqdm(
    range(start_epoch, CFG['gan_epochs']),
    desc    = 'WGAN-GP',
    unit    = 'epoch',
    initial = start_epoch,
    total   = CFG['gan_epochs'],
    position = 0
)

for epoch in epoch_pbar:
    G.train(); D.train()
    epoch_d, epoch_g = [], []

    batch_pbar = tqdm(
        train_loader,
        desc     = f'  Epoch {epoch+1:2d}/{CFG["gan_epochs"]}',
        unit     = 'batch',
        leave    = False,
        position = 1
    )

    for x_real, _, _ in batch_pbar:

        # ── Train Discriminator ───────────────────────────────────────
        for _ in range(CFG['n_critic']):
            opt_D.zero_grad()

            z      = torch.randn(x_real.size(0), CFG['noise_dim'])
            delta  = G(z)
            x_fake = (x_real + delta * CFG['pgd_eps'][1]).detach()
            x_fake = x_fake.clamp(0, 1)

            d_real = D(x_real)
            d_fake = D(x_fake)

            gp     = gradient_penalty(D, x_real, x_fake)
            d_loss = d_fake.mean() - d_real.mean() + CFG['gp_lambda'] * gp

            d_loss.backward()
            opt_D.step()
            epoch_d.append(d_loss.item())

        # ── Train Generator ───────────────────────────────────────────
        opt_G.zero_grad()

        z      = torch.randn(x_real.size(0), CFG['noise_dim'])
        delta  = G(z)
        x_fake = (x_real + delta * CFG['pgd_eps'][1]).clamp(0, 1)

        g_loss = -D(x_fake).mean()
        g_loss.backward()
        opt_G.step()
        epoch_g.append(g_loss.item())

        batch_pbar.set_postfix({
            'D': f'{np.mean(epoch_d):.3f}',
            'G': f'{np.mean(epoch_g):.3f}',
        })

    # ── End of epoch ──────────────────────────────────────────────────
    g_losses.append(np.mean(epoch_g))
    d_losses.append(np.mean(epoch_d))

    elapsed   = time.time() - start_time
    per_epoch = elapsed / (epoch - start_epoch + 1)
    remaining = per_epoch * (CFG['gan_epochs'] - epoch - 1)

    epoch_pbar.set_postfix({
        'D_loss':    f'{d_losses[-1]:.3f}',
        'G_loss':    f'{g_losses[-1]:.3f}',
        'elapsed':   f'{elapsed/60:.1f}m',
        'remaining': f'{remaining/60:.1f}m',
    })

    if (epoch + 1) % 5 == 0:
        print(f"\nEpoch [{epoch+1:3d}/{CFG['gan_epochs']}] "
              f"D_loss: {d_losses[-1]:8.4f} | "
              f"G_loss: {g_losses[-1]:8.4f} | "
              f"Elapsed: {elapsed/60:.1f}m | "
              f"Remaining: {remaining/60:.1f}m")

        # ── Save checkpoint every 5 epochs ────────────────────────────
        ckpt_path = os.path.join(
            checkpoint_dir, f'wgan_epoch_{epoch+1:03d}.pt')
        save_checkpoint(epoch, G, D, opt_G, opt_D,
                        g_losses, d_losses, ckpt_path)

# ── Final save ────────────────────────────────────────────────────────────
print("\nWGAN-GP training complete")
torch.save(G.state_dict(), f"{CFG['output_dir']}/generator.pt")
torch.save(D.state_dict(), f"{CFG['output_dir']}/discriminator.pt")
print("Saved: generator.pt and discriminator.pt")

# Save final losses for plotting later
np.save(f"{CFG['output_dir']}/g_losses.npy", np.array(g_losses))
np.save(f"{CFG['output_dir']}/d_losses.npy", np.array(d_losses))
print("Saved: g_losses.npy and d_losses.npy")

No checkpoint found — starting fresh
Starting from epoch: 1
Remaining epochs:    15

Training WGAN-GP...
Epochs: 15 | Batch: 512
n_critic: 2 | GP lambda: 10
n_features: 66
Checkpoints every 5 epochs → robustlmp_outputs/checkpoints



WGAN-GP:  33%|███▎      | 5/15 [2:38:06<5:17:19, 1903.96s/epoch, D_loss=0.008, G_loss=-0.288, elapsed=158.1m, remaining=316.2m]


Epoch [  5/15] D_loss:   0.0078 | G_loss:  -0.2877 | Elapsed: 158.1m | Remaining: 316.2m
  ✅ Checkpoint saved: robustlmp_outputs/checkpoints/wgan_epoch_005.pt


WGAN-GP:  67%|██████▋   | 10/15 [5:13:35<2:35:55, 1871.00s/epoch, D_loss=0.003, G_loss=-0.913, elapsed=313.6m, remaining=156.8m]


Epoch [ 10/15] D_loss:   0.0031 | G_loss:  -0.9134 | Elapsed: 313.6m | Remaining: 156.8m
  ✅ Checkpoint saved: robustlmp_outputs/checkpoints/wgan_epoch_010.pt


WGAN-GP: 100%|██████████| 15/15 [7:49:41<00:00, 1878.74s/epoch, D_loss=0.004, G_loss=-1.923, elapsed=469.7m, remaining=0.0m]    


Epoch [ 15/15] D_loss:   0.0037 | G_loss:  -1.9234 | Elapsed: 469.7m | Remaining: 0.0m
  ✅ Checkpoint saved: robustlmp_outputs/checkpoints/wgan_epoch_015.pt

WGAN-GP training complete
Saved: generator.pt and discriminator.pt
Saved: g_losses.npy and d_losses.npy


In [113]:
def train_forecaster(train_loader, val_loader,
                     augment=False, tag='baseline'):
    model   = LSTMForecaster(n_features,
                              CFG['lstm_hidden'],
                              CFG['lstm_layers'],
                              CFG['dropout'])
    opt     = torch.optim.Adam(model.parameters(), lr=CFG['lstm_lr'])
    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(
                  opt, patience=3, factor=0.5)  # removed verbose=False
    loss_fn = nn.HuberLoss()

    best_val  = float('inf')
    train_hist, val_hist = [], []

    print(f"\nTraining LSTM [{tag}] | "
          f"Augment: {augment} | "
          f"Epochs: {CFG['lstm_epochs']}")

    for epoch in range(CFG['lstm_epochs']):
        # ── Train ─────────────────────────────────────────────────────
        model.train()
        train_loss = []

        for x_seq, y_rt, y_da in tqdm(train_loader,
                                       desc=f'  Epoch {epoch+1:2d}/{CFG["lstm_epochs"]}',
                                       leave=False):
            if augment:
                n_aug = int(x_seq.size(0) * 0.3)
                if n_aug > 0:
                    z     = torch.randn(n_aug, CFG['noise_dim'])
                    with torch.no_grad():
                        delta = G(z)
                    eps   = CFG['pgd_eps'][0]
                    x_aug = (x_seq[:n_aug] + delta * eps).clamp(0, 1)
                    x_seq = torch.cat([x_seq, x_aug], dim=0)
                    y_rt  = torch.cat([y_rt,  y_rt[:n_aug]],  dim=0)
                    y_da  = torch.cat([y_da,  y_da[:n_aug]],  dim=0)

            opt.zero_grad()
            pred_rt, pred_da = model(x_seq)
            loss = loss_fn(pred_rt, y_rt) + loss_fn(pred_da, y_da)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss.append(loss.item())

        # ── Validate ──────────────────────────────────────────────────
        model.eval()
        val_loss = []
        with torch.no_grad():
            for x_seq, y_rt, y_da in val_loader:
                pred_rt, pred_da = model(x_seq)
                loss = loss_fn(pred_rt, y_rt) + loss_fn(pred_da, y_da)
                val_loss.append(loss.item())

        t_loss = np.mean(train_loss)
        v_loss = np.mean(val_loss)
        train_hist.append(t_loss)
        val_hist.append(v_loss)
        sched.step(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            torch.save(model.state_dict(),
                       f"{CFG['output_dir']}/lstm_{tag}_best.pt")

        if (epoch + 1) % 3 == 0:
            print(f"  Epoch [{epoch+1:2d}/{CFG['lstm_epochs']}] "
                  f"Train: {t_loss:.4f} | Val: {v_loss:.4f} | "
                  f"Best: {best_val:.4f}")

    model.load_state_dict(
        torch.load(f"{CFG['output_dir']}/lstm_{tag}_best.pt"))
    print(f"  Training complete — best val loss: {best_val:.4f}")
    return model, train_hist, val_hist


# ── Run both models ───────────────────────────────────────────────────────
model_base, th_base, vh_base = train_forecaster(
    train_loader, val_loader,
    augment=False, tag='baseline'
)

model_robust, th_robust, vh_robust = train_forecaster(
    train_loader, val_loader,
    augment=True, tag='robust'
)


Training LSTM [baseline] | Augment: False | Epochs: 15


  Epoch [ 3/15] Train: 0.0001 | Val: 0.0001 | Best: 0.0001


  Epoch [ 6/15] Train: 0.0001 | Val: 0.0000 | Best: 0.0000


  Epoch [ 9/15] Train: 0.0001 | Val: 0.0000 | Best: 0.0000


  Epoch [12/15] Train: 0.0000 | Val: 0.0000 | Best: 0.0000


...

  Epoch [ 3/15] Train: 0.0001 | Val: 0.0001 | Best: 0.0001


  Epoch [ 6/15] Train: 0.0001 | Val: 0.0000 | Best: 0.0000


  Epoch [ 9/15] Train: 0.0001 | Val: 0.0000 | Best: 0.0000


  Epoch [12/15] Train: 0.0000 | Val: 0.0000 | Best: 0.0000


  Epoch [15/15] Train: 0.0000 | Val: 0.0000 | Best: 0.0000
  Training complete — best val loss: 0.0000


In [115]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

# ── Define evaluate_model ─────────────────────────────────────────────────
def evaluate_model(model, loader, scaler_rt, scaler_da,
                   use_smoothing=False, attack_eps=None, tag=''):
    model.eval()
    all_rt_true, all_rt_pred = [], []
    all_da_true, all_da_pred = [], []

    for x_seq, y_rt, y_da in tqdm(loader, desc=f'  Evaluating {tag}', leave=False):

        if attack_eps is not None:
            x_seq = pgd_attack(model, x_seq, y_rt, y_da, attack_eps)

        if use_smoothing:
            pred_rt, pred_da = smoothed_predict(model, x_seq)
        else:
            with torch.no_grad():
                pred_rt, pred_da = model(x_seq)

        all_rt_true.append(y_rt.numpy())
        all_rt_pred.append(pred_rt.detach().numpy())
        all_da_true.append(y_da.numpy())
        all_da_pred.append(pred_da.detach().numpy())

    # Inverse scale
    rt_true = scaler_rt.inverse_transform(np.concatenate(all_rt_true))
    rt_pred = scaler_rt.inverse_transform(np.concatenate(all_rt_pred))
    da_true = scaler_da.inverse_transform(np.concatenate(all_da_true))
    da_pred = scaler_da.inverse_transform(np.concatenate(all_da_pred))

    # Metrics
    rt_mape = mean_absolute_percentage_error(rt_true, rt_pred) * 100
    da_mape = mean_absolute_percentage_error(da_true, da_pred) * 100
    rt_rmse = np.sqrt(mean_squared_error(rt_true, rt_pred))
    da_rmse = np.sqrt(mean_squared_error(da_true, da_pred))

    print(f"\n{'='*55}")
    print(f"  {tag}")
    print(f"{'='*55}")
    print(f"  RT-LMP  MAPE: {rt_mape:6.2f}%  |  RMSE: {rt_rmse:6.2f} $/MWh")
    print(f"  DA-LMP  MAPE: {da_mape:6.2f}%  |  RMSE: {da_rmse:6.2f} $/MWh")

    return {
        'tag': tag, 'rt_mape': rt_mape, 'da_mape': da_mape,
        'rt_rmse': rt_rmse, 'da_rmse': da_rmse,
        'rt_true': rt_true, 'rt_pred': rt_pred,
        'da_true': da_true, 'da_pred': da_pred
    }


# ── Define pgd_attack ─────────────────────────────────────────────────────
def pgd_attack(model, x_seq, y_rt, y_da, epsilon,
               alpha=CFG['pgd_alpha'], steps=CFG['pgd_steps']):
    loss_fn = nn.HuberLoss()
    x_adv   = x_seq.clone().detach()
    x_adv   = x_adv + torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
    x_adv   = x_adv.clamp(0, 1)

    for _ in range(steps):
        x_adv.requires_grad_(True)
        pred_rt, pred_da = model(x_adv)
        loss = loss_fn(pred_rt, y_rt) + loss_fn(pred_da, y_da)
        loss.backward()

        with torch.no_grad():
            x_adv = x_adv + alpha * x_adv.grad.sign()
            delta = (x_adv - x_seq).clamp(-epsilon, epsilon)
            x_adv = (x_seq + delta).clamp(0, 1)

    return x_adv.detach()


# ── Define smoothed_predict ───────────────────────────────────────────────
def smoothed_predict(model, x_seq, sigma=CFG['sigma'],
                     n_samples=CFG['n_smooth']):
    model.eval()
    preds_rt, preds_da = [], []

    with torch.no_grad():
        for _ in range(n_samples):
            noise   = torch.randn_like(x_seq) * sigma
            x_noisy = (x_seq + noise).clamp(0, 1)
            p_rt, p_da = model(x_noisy)
            preds_rt.append(p_rt)
            preds_da.append(p_da)

    pred_rt = torch.stack(preds_rt, dim=0).median(dim=0).values
    pred_da = torch.stack(preds_da, dim=0).median(dim=0).values
    return pred_rt, pred_da


print("All evaluation functions defined ✅")

# ── Run all evaluations ───────────────────────────────────────────────────
results = {}

# 1. Baseline — clean
results['base_clean'] = evaluate_model(
    model_base, test_loader, scaler_rt, scaler_da,
    tag='Baseline LSTM — Clean')

# 2. Baseline — under PGD attack
for eps in CFG['pgd_eps']:
    results[f'base_pgd_{eps}'] = evaluate_model(
        model_base, test_loader, scaler_rt, scaler_da,
        attack_eps=eps,
        tag=f'Baseline LSTM — PGD eps={eps}')

# 3. RobustLMP-GAN — clean
results['robust_clean'] = evaluate_model(
    model_robust, test_loader, scaler_rt, scaler_da,
    tag='RobustLMP-GAN — Clean')

# 4. RobustLMP-GAN — under PGD attack
for eps in CFG['pgd_eps']:
    results[f'robust_pgd_{eps}'] = evaluate_model(
        model_robust, test_loader, scaler_rt, scaler_da,
        attack_eps=eps,
        tag=f'RobustLMP-GAN — PGD eps={eps}')

# 5. RobustLMP-GAN + Smoothing — under attack
for eps in CFG['pgd_eps']:
    results[f'smooth_pgd_{eps}'] = evaluate_model(
        model_robust, test_loader, scaler_rt, scaler_da,
        use_smoothing=True, attack_eps=eps,
        tag=f'RobustLMP-GAN+Smooth — PGD eps={eps}')

# ── Print summary table ───────────────────────────────────────────────────
print("\n\nFINAL RESULTS SUMMARY")
print("="*75)
print(f"{'Model':<45} {'RT MAPE':>8} {'DA MAPE':>8} {'RT RMSE':>8} {'DA RMSE':>8}")
print("-"*75)
for key, res in results.items():
    print(f"{res['tag']:<45} "
          f"{res['rt_mape']:>7.2f}% "
          f"{res['da_mape']:>7.2f}% "
          f"{res['rt_rmse']:>7.2f}  "
          f"{res['da_rmse']:>7.2f}")

# ── Save ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame([
    {
        'model':   res['tag'],
        'rt_mape': res['rt_mape'],
        'da_mape': res['da_mape'],
        'rt_rmse': res['rt_rmse'],
        'da_rmse': res['da_rmse'],
    }
    for res in results.values()
])
results_df.to_csv(f"{CFG['output_dir']}/results_summary.csv", index=False)
print(f"\nSaved: results_summary.csv")

All evaluation functions defined ✅



  Baseline LSTM — Clean
  RT-LMP  MAPE:  43.46%  |  RMSE:  17.81 $/MWh
  DA-LMP  MAPE:   9.06%  |  RMSE:   5.11 $/MWh



  Baseline LSTM — PGD eps=0.01
  RT-LMP  MAPE: 171.91%  |  RMSE:  45.01 $/MWh
  DA-LMP  MAPE: 112.72%  |  RMSE:  32.96 $/MWh



  Baseline LSTM — PGD eps=0.05
  RT-LMP  MAPE: 994.65%  |  RMSE: 278.80 $/MWh
  DA-LMP  MAPE: 497.95%  |  RMSE: 141.50 $/MWh



  Baseline LSTM — PGD eps=0.1
  RT-LMP  MAPE: 1675.77%  |  RMSE: 502.38 $/MWh
  DA-LMP  MAPE: 631.03%  |  RMSE: 174.77 $/MWh


...


  RobustLMP-GAN — PGD eps=0.05
  RT-LMP  MAPE: 1914.51%  |  RMSE: 516.10 $/MWh
  DA-LMP  MAPE: 565.05%  |  RMSE: 160.45 $/MWh



  RobustLMP-GAN — PGD eps=0.1
  RT-LMP  MAPE: 2793.76%  |  RMSE: 777.44 $/MWh
  DA-LMP  MAPE: 715.79%  |  RMSE: 198.31 $/MWh



  RobustLMP-GAN+Smooth — PGD eps=0.01
  RT-LMP  MAPE: 229.96%  |  RMSE:  64.11 $/MWh
  DA-LMP  MAPE: 102.32%  |  RMSE:  30.64 $/MWh



  RobustLMP-GAN+Smooth — PGD eps=0.05
  RT-LMP  MAPE: 1499.73%  |  RMSE: 434.81 $/MWh
  DA-LMP  MAPE: 441.12%  |  RMSE: 125.93 $/MWh



  RobustLMP-GAN+Smooth — PGD eps=0.1
  RT-LMP  MAPE: 2478.33%  |  RMSE: 688.53 $/MWh
  DA-LMP  MAPE: 592.74%  |  RMSE: 165.07 $/MWh


FINAL RESULTS SUMMARY
Model                                          RT MAPE  DA MAPE  RT RMSE  DA RMSE
---------------------------------------------------------------------------
Baseline LSTM — Clean                           43.46%    9.06%   17.81     5.11
Baseline LSTM — PGD eps=0.01                   171.91%  112.72%   45.01    32.96
Baseline LSTM — PGD eps=0.05                   994.65%  497.95%  278.80   141.50
Baseline LSTM — PGD eps=0.1                   1675.77%  631.03%  502.38   174.77
RobustLMP-GAN — Clean                           32.04%    9.52%   17.68     5.07
RobustLMP-GAN — PGD eps=0.01                   253.62%  132.58%   69.95    38.96
RobustLMP-GAN — PGD eps=0.05                  1914.51%  565.05%  516.10   160.45
RobustLMP-GAN — PGD eps=0.1                   2793.76%  715.79%  777.44   198.31
RobustLMP-GAN+Smooth — PGD eps=0.01  

# 📊 Model Evaluation Summary

---

## What worked well

### Clean performance

Baseline RT-LMP MAPE: 43.46%

RobustLMP-GAN RT-LMP MAPE: 32.04% ← 26% improvement on clean data ✅


### Day-ahead forecasting

DA-LMP both models: ~9% ← excellent day-ahead forecasting ✅


---

### Randomized smoothing helps at eps=0.01

RobustLMP-GAN PGD: 253.62%

RobustLMP-GAN+Smooth: 229.96% ← smoothing helps ✅


---

# What needs addressing

### Under attack, RobustLMP-GAN is WORSE than baseline

Baseline eps=0.05: 994%

RobustLMP-GAN eps=0.05: 1914% ← worse ❌


This is a known issue — the augmented model learned to be more aggressive on clean data but became more sensitive to larger perturbations

---

# Root cause — 3 issues to fix

---

### Issue 1 — RT-LMP 43% MAPE is too high on clean data

This is because RT prices include extreme spikes (Winter Storm Uri etc).  
We need to clip outliers during training.

---

### Issue 2 — Augmentation epsilon too small

We only augmented with eps=0.01 but tested up to eps=0.1 — the model never saw large perturbations during training.

---

### Issue 3 — Not enough epochs

15 epochs is too few for the robust model to fully converge.



In [ ]:
# ── Fix 1: Clip extreme LMP outliers before scaling ──────────────────────
print("Before clipping:")
print(f"  RT-LMP range: {df_clean['total_lmp_rt'].min():.1f} "
      f"to {df_clean['total_lmp_rt'].max():.1f} $/MWh")
print(f"  RT-LMP 99th pct: {df_clean['total_lmp_rt'].quantile(0.99):.1f}")
print(f"  RT-LMP 1st pct:  {df_clean['total_lmp_rt'].quantile(0.01):.1f}")

# Clip to 1st-99th percentile
rt_low  = df_clean['total_lmp_rt'].quantile(0.01)
rt_high = df_clean['total_lmp_rt'].quantile(0.99)
da_low  = df_clean['total_lmp_da'].quantile(0.01)
da_high = df_clean['total_lmp_da'].quantile(0.99)

df_clean['total_lmp_rt'] = df_clean['total_lmp_rt'].clip(rt_low, rt_high)
df_clean['total_lmp_da'] = df_clean['total_lmp_da'].clip(da_low, da_high)

print(f"\nAfter clipping:")
print(f"  RT-LMP range: {df_clean['total_lmp_rt'].min():.1f} "
      f"to {df_clean['total_lmp_rt'].max():.1f} $/MWh")

# ── Fix 2: Update augmentation to use full epsilon range ─────────────────
# Train with all epsilon levels so model sees large perturbations
CFG['aug_eps_range'] = CFG['pgd_eps']  # [0.01, 0.05, 0.10]

# ── Fix 3: More epochs ────────────────────────────────────────────────────
CFG['lstm_epochs'] = 25

# ── Rebuild train/val/test splits and scalers ─────────────────────────────
train_mask = df_clean['datetime_beginning_ept'] <  '2023-01-01'
val_mask   = (df_clean['datetime_beginning_ept'] >= '2023-01-01') & \
             (df_clean['datetime_beginning_ept'] <  '2023-07-01')
test_mask  = df_clean['datetime_beginning_ept'] >= '2023-07-01'

df_train = df_clean[train_mask].copy()
df_val   = df_clean[val_mask].copy()
df_test  = df_clean[test_mask].copy()

scaler_X  = MinMaxScaler()
scaler_rt = MinMaxScaler()
scaler_da = MinMaxScaler()

X_train = scaler_X.fit_transform(df_train[feature_cols].fillna(0))
X_val   = scaler_X.transform(df_val[feature_cols].fillna(0))
X_test  = scaler_X.transform(df_test[feature_cols].fillna(0))

y_train_rt = scaler_rt.fit_transform(df_train[['total_lmp_rt']])
y_val_rt   = scaler_rt.transform(df_val[['total_lmp_rt']])
y_test_rt  = scaler_rt.transform(df_test[['total_lmp_rt']])

y_train_da = scaler_da.fit_transform(df_train[['total_lmp_da']])
y_val_da   = scaler_da.transform(df_val[['total_lmp_da']])
y_test_da  = scaler_da.transform(df_test[['total_lmp_da']])

# Rebuild loaders
train_ds = LMPDataset(X_train, y_train_rt, y_train_da, CFG['seq_len'])
val_ds   = LMPDataset(X_val,   y_val_rt,   y_val_da,   CFG['seq_len'])
test_ds  = LMPDataset(X_test,  y_test_rt,  y_test_da,  CFG['seq_len'])

train_loader = DataLoader(train_ds, batch_size=CFG['lstm_batch'],
                          shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['lstm_batch'],
                          shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=CFG['lstm_batch'],
                          shuffle=False, drop_last=False)

print(f"\nRebuilt loaders with clipped targets")
print(f"Train batches: {len(train_loader)}")

# ── Updated train_forecaster with full epsilon range augmentation ─────────
def train_forecaster(train_loader, val_loader,
                     augment=False, tag='baseline'):
    model   = LSTMForecaster(n_features,
                              CFG['lstm_hidden'],
                              CFG['lstm_layers'],
                              CFG['dropout'])
    opt     = torch.optim.Adam(model.parameters(), lr=CFG['lstm_lr'])
    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(
                  opt, patience=3, factor=0.5)
    loss_fn = nn.HuberLoss()

    best_val   = float('inf')
    train_hist, val_hist = [], []

    print(f"\nTraining LSTM [{tag}] | "
          f"Augment: {augment} | "
          f"Epochs: {CFG['lstm_epochs']}")

    for epoch in range(CFG['lstm_epochs']):
        model.train()
        train_loss = []

        for x_seq, y_rt, y_da in tqdm(train_loader,
                                       desc=f'  Epoch {epoch+1:2d}/{CFG["lstm_epochs"]}',
                                       leave=False):
            if augment:
                n_aug = int(x_seq.size(0) * 0.3)
                if n_aug > 0:
                    # ── Use random epsilon from full range ────────────
                    eps = float(np.random.choice(CFG['aug_eps_range']))
                    z   = torch.randn(n_aug, CFG['noise_dim'])
                    with torch.no_grad():
                        delta = G(z)
                    x_aug = (x_seq[:n_aug] + delta * eps).clamp(0, 1)
                    x_seq = torch.cat([x_seq, x_aug], dim=0)
                    y_rt  = torch.cat([y_rt,  y_rt[:n_aug]], dim=0)
                    y_da  = torch.cat([y_da,  y_da[:n_aug]], dim=0)

            opt.zero_grad()
            pred_rt, pred_da = model(x_seq)
            loss = loss_fn(pred_rt, y_rt) + loss_fn(pred_da, y_da)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss.append(loss.item())

        model.eval()
        val_loss = []
        with torch.no_grad():
            for x_seq, y_rt, y_da in val_loader:
                pred_rt, pred_da = model(x_seq)
                loss = loss_fn(pred_rt, y_rt) + loss_fn(pred_da, y_da)
                val_loss.append(loss.item())

        t_loss = np.mean(train_loss)
        v_loss = np.mean(val_loss)
        train_hist.append(t_loss)
        val_hist.append(v_loss)
        sched.step(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            torch.save(model.state_dict(),
                       f"{CFG['output_dir']}/lstm_{tag}_best.pt")

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch [{epoch+1:2d}/{CFG['lstm_epochs']}] "
                  f"Train: {t_loss:.4f} | Val: {v_loss:.4f} | "
                  f"Best: {best_val:.4f}")

    model.load_state_dict(
        torch.load(f"{CFG['output_dir']}/lstm_{tag}_best.pt"))
    print(f"  Training complete — best val loss: {best_val:.4f}")
    return model, train_hist, val_hist


# ── Retrain both models ───────────────────────────────────────────────────
model_base, th_base, vh_base = train_forecaster(
    train_loader, val_loader,
    augment=False, tag='baseline'
)

model_robust, th_robust, vh_robust = train_forecaster(
    train_loader, val_loader,
    augment=True, tag='robust'
)

Before clipping:
  RT-LMP range: -126.3 to 4360.9 $/MWh
  RT-LMP 99th pct: 150.6
  RT-LMP 1st pct:  6.6

After clipping:
  RT-LMP range: 6.6 to 150.6 $/MWh

Rebuilt loaders with clipped targets
Train batches: 800

Training LSTM [baseline] | Augment: False | Epochs: 25


  Epoch [ 5/25] Train: 0.0044 | Val: 0.0039 | Best: 0.0036


  Epoch [10/25] Train: 0.0038 | Val: 0.0029 | Best: 0.0029


  Epoch [15/25] Train: 0.0035 | Val: 0.0026 | Best: 0.0026


  Epoch [20/25] Train: 0.0031 | Val: 0.0026 | Best: 0.0026


  Epoch [25/25] Train: 0.0029 | Val: 0.0026 | Best: 0.0026
  Training complete — best val loss: 0.0026

Training LSTM [robust] | Augment: True | Epochs: 25


  Epoch  5/25:  58%|█████▊    | 465/800 [00:54<00:43,  7.63it/s]